# sLDSC heritability enrichment and tau* for fine-mapped xQTL annotations


## Setup


In [1]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(stringr)
  library(data.table)
  library(ggplot2)
  library(ComplexHeatmap)
  library(circlize)
  library(grid)
  library(RColorBrewer)
})

# Output directories: 2_single_context_cis figures stay next to this notebook;
# coloc+multi-context landmark figures go to 3_multi_context/figure_data.
figure_data_dir       <- normalizePath("../figure_data", mustWork = FALSE)
figure_out_dir        <- normalizePath(file.path(".", "figure_data"), mustWork = FALSE)
figure_data_multi_dir <- normalizePath("../../3_multi_context/figure_data",
                                       mustWork = FALSE)
dir.create(figure_data_dir,       showWarnings = FALSE, recursive = TRUE)
dir.create(figure_out_dir,        showWarnings = FALSE, recursive = TRUE)
dir.create(figure_data_multi_dir, showWarnings = FALSE, recursive = TRUE)

sldsc_path <- file.path(figure_data_dir, "Figure_2_sLDSC_FM_all_methods.rds")

save_heatmap_pdf <- function(ht, full_path, width, height,
                              extra_legend = NULL) {
  pdf(full_path, width = width, height = height)
  draw(ht, heatmap_legend_side = "right",
       annotation_legend_side = "right",
       heatmap_legend_list = extra_legend,
       merge_legend = TRUE,
       padding = unit(c(2, 2, 2, 2), "mm"))
  dev.off()
  message("Saved: ", full_path)
  invisible(full_path)
}


## Data Preparation

This section regenerates the cached RDS file used downstream:

- `sldsc_path` (`Figure_2_sLDSC_FM_all_methods.rds`) bundles, for the FM sLDSC
  run (yl4437 step2/output):
  - `enr_df`  : per-study x trait sLDSC enrichment summary
    (`Enrichment`, `Enrichment_p`, `Prop._h2`, `Prop._SNPs`, ...).
  - `tra_df`  : trait -> Category map.
  - `loci_df` : study -> method / loci_type / loci_group
    (method = `univariate` / `colocboost` / `multicontext`).
  - `tau_df`  : per-study x trait `tau*` summary
    (`tau_star`, `tau_star_se`, `h2g`, `tau_z`).

The block is wrapped in `if (FORCE_REPREP || !file.exists(...))` so a re-run
will skip recomputation when the cached RDS exists.

Inputs (per-study `.single_tau.initial_processed_stats.rds`):
- `/mnt/lustre/home/yl4437/work/xqtl_flagship/flagship_paper_check/sLDSC/single_tau/step2/output/{study}/all_m_rd5/{study}.single_tau.initial_processed_stats.rds`

Method is derived from the FM folder name:
- `*Multi_All` -> `multicontext`
- `*colocboost*` -> `colocboost`
- otherwise -> `univariate`

`*_multigene` study folders are excluded at discovery.
tau* = tau_c x sd_c x M_ref / h2g (Finucane et al. 2018), extracted from
`sc_matrix` (200-block jackknife): mean = point estimate; SE = sqrt(B-1) x sd.


In [2]:
# Inputs
sldsc_fm_dir <- "/mnt/lustre/home/yl4437/work/xqtl_flagship/flagship_paper_check/sLDSC/single_tau/step2/output"

# Force regeneration by setting FORCE_REPREP <- TRUE before running
FORCE_REPREP <- if (exists("FORCE_REPREP")) FORCE_REPREP else FALSE

# Trait -> Category map (covers AD/Blood/Autoimmune/Neurodegenerative/Behavioral/
# Neuroimaging/Psychiatric/Other GWAS traits used in the FM panels).
category_map <- c(
  # AD
  "AD_Bellenguez_EADB_hg38_sorted" = "AD",
  "AD_Bellenguez_buildGRCh38" = "AD",
  "AD_Kunkle_etal_Stage1_results_hg38.sorted.munged.parquet" = "AD",
  "AD_Kunkle_etal_Stage1_results_hg38.sorted.munged" = "AD",
  "AD_Wightman_Excluding23andMe" = "AD",
  "AD_Wightman_ExcludingUKB23andMe" = "AD",
  "AD_Wightman_Excluding23andMe_hg38_sorted" = "AD",
  "AD_Wightman_ExcludingUKB23andMe_hg38_sorted" = "AD",
  "AD_Wightman_full_2021" = "AD",
  "PASS_Alzheimers_Jansen2019" = "AD",
  # Blood
  "UKB.Baso.BOLT" = "Blood", "UKB.Eosino.BOLT" = "Blood",
  "UKB.Lym.BOLT" = "Blood", "UKB.MCV.BOLT" = "Blood",
  "UKB.Mono.BOLT" = "Blood", "UKB.Neutro.BOLT" = "Blood",
  "UKB.Plt.BOLT" = "Blood", "UKB.RBC.BOLT" = "Blood",
  "UKB_460K.blood_RBC_DISTRIB_WIDTH" = "Blood",
  "UKB_460K.biochemistry_AlkalinePhosphatase" = "Blood",
  "UKB_460K.biochemistry_AspartateAminotransferase" = "Blood",
  "UKB_460K.biochemistry_Creatinine" = "Blood",
  "UKB.TBil.BOLT" = "Blood", "UKB.TP.BOLT" = "Blood",
  "UKB.VitD.BOLT" = "Blood",
  # Autoimmune
  "PASS_CD_deLange2017" = "Autoimmune", "PASS_Celiac" = "Autoimmune",
  "PASS_IBD_deLange2017" = "Autoimmune", "PASS_Lupus" = "Autoimmune",
  "PASS_Multiple_sclerosis" = "Autoimmune",
  "PASS_Primary_biliary_cirrhosis" = "Autoimmune",
  "PASS_Rheumatoid_Arthritis" = "Autoimmune",
  "PASS_Type_1_Diabetes" = "Autoimmune",
  "PASS_UC_deLange2017" = "Autoimmune",
  "UKB.AID_Combined.SAIGE" = "Autoimmune",
  "UKB.Hypothyroidism.SAIGE" = "Autoimmune",
  "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED" = "Autoimmune",
  "PASS_ChildOnsetAsthma_Ferreira2019" = "Autoimmune",
  "MS" = "Autoimmune",
  # Neurodegenerative
  "Dementia_Chia_GCST90001390_buildGRCh38_sorted_cleaned.munged" = "Neurodegenerative",
  "FTD_GWAS_META_sorted_cleaned.munged" = "Neurodegenerative",
  "ALS-Rheenen-2021.GCST90027163_buildGRCh37" = "Neurodegenerative",
  "ALS-Rheenen-2021.GCST90027164_buildGRCh37" = "Neurodegenerative",
  "nallsEtAl2019_excluding23andMe" = "Neurodegenerative",
  "MegaGWAS_summary_European" = "Neurodegenerative",
  # Behavioral
  "EXF"="Behavioral","LAN"="Behavioral","MEM"="Behavioral",
  "PASS_AgeFirstBirth"="Behavioral","PASS_Intelligence_SavageJansen2018"="Behavioral",
  "PASS_ReactionTime_Davies2018"="Behavioral","PASS_SleepDuration_Dashti2019"="Behavioral",
  "UKB.BMI.BOLT"="Behavioral","UKB.Insomnia.BOLT"="Behavioral",
  "UKB.Morning_Person.BOLT"="Behavioral","UKB.Smoking_Ever_Never.SAIGE"="Behavioral",
  "UKB_460K.cov_EDU_YEARS"="Behavioral","UKB_460K.repro_NumberChildrenEverBorn_Pooled"="Behavioral",
  "GWAS_CP_all"="Behavioral","GWAS_EA_excl23andMe"="Behavioral",
  "PASS_GeneralRiskTolerance_KarlssonLinner2019"="Behavioral",
  "PASS_Height1"="Behavioral","UKB.WHRadjBMI.BOLT"="Behavioral",
  # Neuroimaging
  "accumbens"="Neuroimaging","amygdala"="Neuroimaging","brainstem"="Neuroimaging",
  "caudate"="Neuroimaging","image_AD1"="Neuroimaging","image_AD2"="Neuroimaging",
  "image_Aging1"="Neuroimaging","image_Aging2"="Neuroimaging","image_Aging3"="Neuroimaging",
  "image_Aging4"="Neuroimaging","image_Aging5"="Neuroimaging","pallidum"="Neuroimaging",
  "putamen"="Neuroimaging","rmf.surf"="Neuroimaging","rmf.thick"="Neuroimaging",
  "sf.surf"="Neuroimaging","sf.thick"="Neuroimaging","mvAge_hg38_sorted_munged.tsv"="Neuroimaging",
  # Psychiatric
  "PASS_ADHD_Demontis2018"="Psychiatric","PASS_Anorexia"="Psychiatric",
  "PASS_Autism"="Psychiatric","PASS_BipolarDisorder_Ruderfer2018"="Psychiatric",
  "PASS_MDD_Wray2018"="Psychiatric","PASS_Neuroticism"="Psychiatric",
  "PASS_SCZvsBD_Ruderfer2018"="Psychiatric","PASS_Schizophrenia"="Psychiatric",
  "PGC3_SCZ_wave3_public.v2"="Psychiatric","PGC_UKB_23andMe_depression_10000"="Psychiatric",
  "PGC_UKB_depression_genome-wide"="Psychiatric","neuroticism_ctg"="Psychiatric",
  # Other
  "CAD_META.filtered"="Other","Mahajan.NatGenet2018b.T2D.European"="Other",
  "TL1_hg38_sorted_munged.tsv"="Other","UKB_460K.bp_DIASTOLICadjMEDz"="Other",
  "UKB_460K.bp_SYSTOLICadjMEDz"="Other","UKB_460K.impedance_BASAL_METABOLIC_RATEz"="Other",
  "UKB.BrC.SAIGE"="Other","UKB.FEV1FVC.BOLT"="Other"
)

dataset_to_trait <- function(ds) {
  x <- sub("_munged\\.parquet$", "", ds)
  x <- sub("\\.sumstats\\.parquet$", "", x)
  sub("\\.parquet$", "", x)
}


In [3]:
# Build Figure_2_sLDSC_FM_all_methods.rds
if (FORCE_REPREP || !file.exists(sldsc_path)) {

  # 1) Discover FM study folders (skip top-level all_m_rd5 placeholder + multigene)
  fm_study_dirs <- list.dirs(sldsc_fm_dir, recursive = FALSE, full.names = TRUE)
  fm_study_dirs <- fm_study_dirs[basename(fm_study_dirs) != "all_m_rd5"]
  fm_study_dirs <- fm_study_dirs[!grepl("_multigene$", basename(fm_study_dirs))]
  fm_studies    <- basename(fm_study_dirs)
  message("[sldsc] FM study folders (multigene excluded): ", length(fm_studies))

  # 2) Classify method and QTL type
  get_fm_method <- function(s) {
    if (grepl("Multi_All$", s)) return("multicontext")
    if (grepl("colocboost", s)) return("colocboost")
    "univariate"
  }
  get_fm_qtl_type <- function(s) {
    if (grepl("_eQTL",  s)) return("eQTL")
    if (grepl("_pQTL",  s)) return("pQTL")
    if (grepl("_mQTL",  s)) return("mQTL")
    if (grepl("_sQTL",  s)) return("sQTL")
    if (grepl("_haQTL", s)) return("haQTL")
    if (grepl("_gpQTL", s)) return("gpQTL")
    if (grepl("_trans_", s)) return("trans")
    "other"
  }
  fm_method   <- vapply(fm_studies, get_fm_method,   character(1))
  fm_qtl_type <- vapply(fm_studies, get_fm_qtl_type, character(1))
  loci_df <- data.table(
    study      = fm_studies,
    method     = fm_method,
    loci_type  = fm_qtl_type,
    loci_group = paste0(fm_method, "_", fm_qtl_type)
  )
  message("[sldsc] method breakdown:")
  print(loci_df[, .N, by = .(method, loci_group)][order(method)])

  # 3) Read enrichment + tau* from each study's single_tau RDS
  fm_rds <- file.path(sldsc_fm_dir, fm_studies, "all_m_rd5",
                      paste0(fm_studies, ".single_tau.initial_processed_stats.rds"))
  fm_rds_ok <- file.exists(fm_rds)
  message("[sldsc] FM RDS found: ", sum(fm_rds_ok), " / ", length(fm_rds))

  read_one_study <- function(rds_path, study_name) {
    se1 <- readRDS(rds_path)
    enr <- bind_rows(lapply(names(se1), function(nm) {
      tmp <- se1[[nm]][["enrichment"]][["enrichment_summary"]]
      if (is.null(tmp)) return(NULL)
      tmp$dataset <- nm
      tmp
    }))
    tau <- bind_rows(lapply(names(se1), function(nm) {
      st <- se1[[nm]]$single_tau
      if (is.null(st) || is.null(st$sc_matrix)) return(NULL)
      x <- as.numeric(st$sc_matrix); x <- x[is.finite(x)]
      B <- length(x); if (B < 2) return(NULL)
      data.frame(
        tau_star    = mean(x, na.rm = TRUE),
        tau_star_se = sqrt(B - 1) * sd(x, na.rm = TRUE),
        h2g         = st$h2g,
        n_jackknife = B,
        dataset     = nm,
        stringsAsFactors = FALSE
      )
    }))
    if (nrow(enr) > 0L) enr$study <- study_name
    if (nrow(tau) > 0L) tau$study <- study_name
    list(enr = enr, tau = tau)
  }

  per_study <- mapply(function(rp, sn) {
    if (!file.exists(rp)) { message("MISSING: ", rp); return(list(enr = NULL, tau = NULL)) }
    read_one_study(rp, sn)
  }, fm_rds, fm_studies, SIMPLIFY = FALSE)

  enr_df <- as.data.table(bind_rows(lapply(per_study, `[[`, "enr")))
  tau_df <- as.data.table(bind_rows(lapply(per_study, `[[`, "tau")))

  # Fix doubled column names from single_tau enrichment_summary
  colnames(enr_df)[1:4] <- c("Enrichment", "Enrichment_std_error",
                              "Prop._h2", "Prop._SNPs")
  enr_df[, trait := dataset_to_trait(dataset)]
  tau_df[, trait := dataset_to_trait(dataset)]
  tau_df[, tau_z := ifelse(is.finite(tau_star_se) & tau_star_se > 0,
                            tau_star / tau_star_se, NA_real_)]
  # Deduplicate any (study, trait) collisions (rare; harmless to keep first)
  enr_df <- enr_df[!duplicated(enr_df[, .(study, trait)])]
  tau_df <- tau_df[!duplicated(tau_df[, .(study, trait)])]

  message("[sldsc] enr_df: ", nrow(enr_df), " rows | ",
          length(unique(enr_df$study)), " studies | ",
          length(unique(enr_df$trait)), " traits")
  message("[sldsc] tau_df: ", nrow(tau_df), " rows | ",
          length(unique(tau_df$study)), " studies | ",
          length(unique(tau_df$trait)), " traits")

  # 4) tra_df: trait -> Category
  tra_df <- unique(enr_df[, .(trait, dataset)])[, category := category_map[trait]]
  unclassified <- tra_df[is.na(category), unique(trait)]
  if (length(unclassified))
    message("[sldsc] unclassified traits (", length(unclassified), "): ",
            paste(head(unclassified, 10), collapse = ", "))

  out <- list(
    data = list(
      enr_df  = enr_df,
      tra_df  = tra_df,
      loci_df = loci_df,
      tau_df  = tau_df
    ),
    meta = list(
      input_dir   = sldsc_fm_dir,
      tau_def     = "mean over 200-block jackknife sc_matrix; SE = sqrt(B-1)*sd",
      method_def  = "from folder name: Multi_All -> multicontext; *colocboost* -> colocboost; else univariate",
      exclusions  = "_multigene study folders dropped at discovery",
      generated_at = format(Sys.time(), "%Y-%m-%d %H:%M:%S")
    )
  )
  saveRDS(out, sldsc_path)
  message("[sldsc] saved: ", sldsc_path)
  invisible(out)
} else {
  message("[sldsc] cache hit: ", sldsc_path)
}


[sldsc] FM study folders (multigene excluded): 64



[sldsc] method breakdown:



         method         loci_group     N
         <char>             <char> <int>
1:   colocboost   colocboost_other     4
2: multicontext multicontext_other     2
3:   univariate    univariate_eQTL    40
4:   univariate    univariate_mQTL     3
5:   univariate    univariate_pQTL     3
6:   univariate   univariate_gpQTL     2
7:   univariate   univariate_haQTL     1
8:   univariate    univariate_sQTL     3
9:   univariate   univariate_trans     6


[sldsc] FM RDS found: 64 / 64



[sldsc] enr_df: 6144 rows | 64 studies | 96 traits



[sldsc] tau_df: 6144 rows | 64 studies | 96 traits



[sldsc] saved: /mnt/lustre/lab/gwang/users/rf2872/Work/codes/xqtl-paper/main_text/2_single_context_cis/figure_data/Figure_2_sLDSC_FM_all_methods.rds



## Main Figure Data and Helpers


In [4]:
Figure_2_sLDSC <- readRDS(sldsc_path)

enr_fm_all  <- as_tibble(Figure_2_sLDSC$data$enr_df) %>%
  mutate(trait = sub("\\.parquet$", "", trait))
tra_fm_all  <- as_tibble(Figure_2_sLDSC$data$tra_df) %>%
  mutate(trait = sub("\\.parquet$", "", trait)) %>%
  distinct(trait, .keep_all = TRUE)
loci_fm_all <- as_tibble(Figure_2_sLDSC$data$loci_df)
tau_fm_all  <- as_tibble(Figure_2_sLDSC$data$tau_df) %>%
  mutate(trait = sub("\\.parquet$", "", trait)) %>%
  distinct(study, trait, .keep_all = TRUE)

message("[plot] enr studies = ", length(unique(enr_fm_all$study)),
        " | traits = ", length(unique(enr_fm_all$trait)),
        " | tau studies = ", length(unique(tau_fm_all$study)),
        " | tau traits = ",  length(unique(tau_fm_all$trait)))

# ── Shared transformations & label helpers ──────────────────────────────────
signed_log2 <- function(x) sign(x) * log2(abs(x) + 1)

prettify_labels <- function(x) {
  x <- gsub("DeJager", "CUIMC1", x, fixed = TRUE)
  x <- gsub("Kellis",  "MIT",    x, fixed = TRUE)
  x <- gsub("_eQTL_brain", " brain", x, fixed = TRUE)
  x <- gsub("_eQTL_Mac",   " Mac",   x, fixed = TRUE)
  x <- gsub("_eQTL$",      "",       x, perl  = TRUE)
  x <- gsub("nallsEtAl2019",    "Nalls etAl 2019",  x, fixed = TRUE)
  x <- gsub("ALS-Rheenen-2021", "ALS Rheenen 2021", x, fixed = TRUE)
  x <- gsub("Stage1",    "Stage 1", x, fixed = TRUE)
  x <- gsub("Type_1",    "Type1",   x, fixed = TRUE)
  x <- gsub("blood_RBC_DISTRIB_WIDTH", "RBC Distribution Width", x, fixed = TRUE)
  x <- gsub("_DIAGNOSED", "", x, fixed = TRUE)
  x <- gsub("([A-Za-z])([0-9]{4})\\b", "\\1 \\2", x, perl = TRUE)
  for (pat in c("\\.sumstats\\.parquet$", "\\.parquet$",
                "_munged", "\\.munged", "_sorted", "\\.sorted",
                "_buildGRCh3[78]", "\\.buildGRCh3[78]",
                "_hg38", "\\.hg38", "_hg19", "\\.hg19",
                "_results", "\\.results", "_etal", "\\.etal"))
    x <- gsub(pat, "", x, perl = TRUE)
  x <- trimws(gsub("\\s+", " ",
                    gsub("[_.]+", " ", x, perl = TRUE), perl = TRUE))
  canonical <- c(
    "pass"="PASS","ukb"="UKB","ad"="AD","als"="ALS","aid"="AID",
    "saige"="SAIGE","bolt"="BOLT","mit"="MIT","cuimc1"="CUIMC1",
    "dlpfc"="DLPFC","starnet"="STARNET","rosmap"="ROSMAP",
    "mcv"="MCV","rbc"="RBC","eadb"="EADB","pcc"="PCC",
    "opc"="OPC","ms"="MS","cd"="CD","ibd"="IBD","uc"="UC",
    "ac"="AC","ast"="Ast","exc"="Exc","inh"="Inh","mic"="Mic","oli"="Oli","mac"="Mac",
    "delange"="deLange","23andme"="23andMe",
    "excluding23andme"="Excluding23andMe","excludingukb23andme"="ExcludingUKB23andMe",
    "etal"="et al","eqtl"="eQTL","type1"="Type 1"
  )
  connectors <- c("and","of","the","for","in","or")
  tok <- function(tk, first) {
    if (!nzchar(tk)) return(tk)
    key <- tolower(tk)
    if (key %in% names(canonical)) return(canonical[[key]])
    if (key %in% connectors)
      return(if (first) paste0(toupper(substr(tk,1,1)),
                                tolower(substr(tk,2,nchar(tk))))
              else tolower(tk))
    if (grepl("^(GCST[0-9]+|[0-9]+K|[0-9]+)$", tk, perl = TRUE)) return(toupper(tk))
    if (grepl("[A-Z]", substr(tk, 2, nchar(tk)))) return(tk)
    paste0(toupper(substr(tk,1,1)), tolower(substr(tk,2,nchar(tk))))
  }
  vapply(x, function(s) {
    if (!nzchar(s)) return(s)
    tks <- strsplit(s, " ", fixed = TRUE)[[1]]
    paste(mapply(tok, tks, seq_along(tks) == 1L), collapse = " ")
  }, character(1), USE.NAMES = FALSE)
}

recode_method <- function(m) {
  dplyr::recode(m, colocboost = "ColocBoost", multicontext = "Multi-ctx",
                .default = as.character(m))
}
clean_qtl_row_labels <- function(x) {
  y <- dplyr::case_when(
    grepl("^ROSMAP_eQTL_", x)  ~ paste(sub("^ROSMAP_eQTL_",  "", x), "eQTL"),
    grepl("^ROSMAP_sQTL_", x)  ~ paste(sub("^ROSMAP_sQTL_",  "", x), "sQTL"),
    grepl("^ROSMAP_pQTL_", x)  ~ "pQTL",
    grepl("^ROSMAP_gpQTL_", x) ~ "gpQTL",
    grepl("^ROSMAP_haQTL_", x) ~ "haQTL",
    grepl("^ROSMAP_mQTL_", x)  ~ "mQTL",
    TRUE ~ x
  )
  y <- gsub("_Klein|_Bennett|_mega|_unadjusted|_adjusted", "", y)
  y <- gsub("_", " ", y)
  y <- gsub("monocyte", "Mono", y, ignore.case = TRUE)
  trimws(gsub("\\s+", " ", y, perl = TRUE))
}


# ── Color scales & significance helpers ────────────────────────────────────
make_category_palette <- function(lvls) {
  base <- c("#0072B2","#009E73","#D55E00","#CC79A7","#E69F00",
            "#56B4E9","#882255","#117733")
  cols <- if (length(lvls) <= length(base)) base[seq_along(lvls)]
          else colorRampPalette(base)(length(lvls))
  setNames(cols, lvls)
}

# Enrichment: desaturated diverging palette on signed log2 scale.
col_fun_opt1 <- colorRamp2(
  signed_log2(c(-1000,-300,-100,-10,0,10,100,300,1000)),
  c("#7A9DB5","#A0BDD0","#C8D7E2","#E5EDF3","#FFFFFF",
    "#F4A582","#D6604D","#B2182B","#67001F")
)
# tau*: bounded linear palette (FM-only).
tau_col_fun_fm <- colorRamp2(
  c(-2.0, -1.0, 0, 0.5, 1.0, 2.0),
  c("#053061","#2166AC","#FFFFFF","#F4A582","#D6604D","#67001F")
)

star_from_p <- function(p) {
  if (is.na(p)) return("")
  if (p < 0.001) "***" else if (p < 0.01) "**" else if (p < 0.05) "*" else ""
}
star_from_z <- function(z) {
  if (is.na(z)) return("")
  az <- abs(z)
  if (az > 4.4) "***" else if (az > 3.3) "**" else if (az > 1.96) "*" else ""
}


[plot] enr studies = 64 | traits = 96 | tau studies = 64 | tau traits = 96



## Supplementary Figure: All-method FM enrichment + tau*

Renders 4 PDFs into `figure_data/`:
- `Supplementary_SLDSC_FM_all_methods_enr_all.pdf`  (all traits, enrichment)
- `Supplementary_SLDSC_FM_all_methods_enr_main.pdf` (Behav/Neuroimg/Psych dropped, enrichment)
- `Supplementary_SLDSC_FM_all_methods_tau_all.pdf`  (all traits, tau*)
- `Supplementary_SLDSC_FM_all_methods_tau_main.pdf` (Behav/Neuroimg/Psych dropped, tau*)


In [5]:
loci_g1    <- loci_fm_all %>% mutate(method = recode_method(method))
methods_g1 <- sort(unique(loci_g1$method))

enr_g1 <- enr_fm_all %>%
  left_join(tra_fm_all %>% distinct(trait, category), by = "trait") %>%
  left_join(loci_g1   %>% select(study, method, loci_group), by = "study") %>%
  distinct(study, trait, .keep_all = TRUE)

tau_g1 <- tau_fm_all %>%
  left_join(tra_fm_all %>% distinct(trait, category), by = "trait") %>%
  left_join(loci_g1   %>% select(study, method, loci_group), by = "study") %>%
  distinct(study, trait, .keep_all = TRUE)

pivot_mat <- function(df, value_col, studies, traits) {
  m <- df %>% select(study, trait, all_of(value_col)) %>%
    distinct(study, trait, .keep_all = TRUE) %>%
    pivot_wider(names_from = trait, values_from = all_of(value_col)) %>%
    column_to_rownames("study") %>% as.matrix()
  miss <- setdiff(traits, colnames(m))
  if (length(miss)) {
    pad <- matrix(NA_real_, nrow = nrow(m), ncol = length(miss),
                  dimnames = list(rownames(m), miss))
    m <- cbind(m, pad)
  }
  miss_r <- setdiff(studies, rownames(m))
  if (length(miss_r)) {
    pad <- matrix(NA_real_, nrow = length(miss_r), ncol = ncol(m),
                  dimnames = list(miss_r, colnames(m)))
    m <- rbind(m, pad)
  }
  m[studies, traits, drop = FALSE]
}

render_g1 <- function(drop_cats, suffix) {
  df_e <- if (length(drop_cats)) enr_g1 %>% filter(!category %in% drop_cats)
          else enr_g1
  if (nrow(df_e) == 0L) stop("no enrichment rows after category filter")

  m_enr_x <- df_e %>% select(study, trait, Enrichment) %>%
    pivot_wider(names_from = trait, values_from = Enrichment) %>%
    arrange(study) %>% column_to_rownames("study") %>% as.matrix()
  m_p_x <- df_e %>% select(study, trait, Enrichment_p) %>%
    pivot_wider(names_from = trait, values_from = Enrichment_p) %>%
    arrange(study) %>% column_to_rownames("study") %>% as.matrix()
  m_p_x <- m_p_x[rownames(m_enr_x),
                 colnames(m_enr_x)[colnames(m_enr_x) %in% colnames(m_p_x)],
                 drop = FALSE]

  cat_full <- c("AD","Blood","Autoimmune","Neurodegenerative",
                "Behavioral","Neuroimaging","Psychiatric","Other")
  col_cat_x <- tibble(trait = colnames(m_enr_x)) %>%
    left_join(tra_fm_all %>% distinct(trait, category), by = "trait")
  col_cat_x$category[is.na(col_cat_x$category)] <- "Unknown"
  cat_use <- intersect(cat_full, unique(col_cat_x$category))
  extra <- setdiff(unique(col_cat_x$category), cat_use)
  if (length(extra)) cat_use <- c(cat_use, extra)

  all_cols_x <- unlist(lapply(cat_use, function(ct) {
    idx <- which(col_cat_x$category == ct)
    if (length(idx) == 0L) return(character(0))
    if (length(idx) == 1L) return(col_cat_x$trait[idx])
    sub_m <- m_enr_x[, idx, drop = FALSE]; sub_m[is.na(sub_m)] <- 0
    col_cat_x$trait[idx][hclust(dist(t(sub_m)), method = "complete")$order]
  }))
  m_enr_x <- m_enr_x[, all_cols_x, drop = FALSE]
  m_p_x   <- m_p_x[, all_cols_x[all_cols_x %in% colnames(m_p_x)], drop = FALSE]

  studies_x <- rownames(m_enr_x)
  traits_x  <- colnames(m_enr_x)

  m_tau_x <- pivot_mat(tau_g1, "tau_star", studies_x, traits_x)
  m_z_x   <- pivot_mat(tau_g1, "tau_z",    studies_x, traits_x)

  rm_x  <- loci_g1 %>% distinct(study, method)
  rs_x  <- factor(rm_x$method[match(studies_x, rm_x$study)],
                  levels = methods_g1)
  message(sprintf("[g1:%s] %d studies x %d traits", suffix,
                   length(studies_x), length(traits_x)))
  print(table(rs_x))

  ca_x <- tibble(trait = traits_x) %>%
    left_join(tra_fm_all %>% distinct(trait, category), by = "trait")
  ca_x$category[is.na(ca_x$category)] <- "Unknown"
  ca_x$category <- factor(ca_x$category, levels = cat_use)
  cat_col_x <- make_category_palette(levels(ca_x$category))

  ha_top_x <- HeatmapAnnotation(
    Bar = anno_block(gp = gpar(fill = cat_col_x, col = NA),
                     height = unit(3, "mm")),
    show_annotation_name = FALSE
  )
  cat_legend_x <- Legend(
    labels = levels(ca_x$category), legend_gp = gpar(fill = cat_col_x),
    title = "Category", title_gp = gpar(fontsize = 11, fontface = "bold"),
    labels_gp = gpar(fontsize = 10),
    grid_height = unit(4, "mm"), grid_width = unit(4, "mm")
  )

  make_ht <- function(value_mat, sig_mat, display_mat, col_fun_use,
                      leg_title, leg_at, leg_labels, white_thr, star_fn) {
    n_t <- ncol(display_mat); n_s <- nrow(display_mat)
    Heatmap(
      display_mat, name = leg_title, col = col_fun_use, na_col = "grey90",
      width  = unit(n_t * 7, "mm"),
      height = unit(n_s * 7, "mm"),
      cluster_rows = FALSE, cluster_columns = FALSE,
      show_row_dend = FALSE, show_column_dend = FALSE,
      row_split = rs_x, cluster_row_slices = FALSE,
      column_split = ca_x$category, cluster_column_slices = FALSE,
      top_annotation = ha_top_x,
      row_labels     = clean_qtl_row_labels(rownames(display_mat)),
      column_labels  = prettify_labels(colnames(display_mat)),
      show_row_names = TRUE,
      row_title_gp   = gpar(fontsize = 12, fontface = "bold"),
      row_title_rot  = 90,
      column_title   = NULL,
      row_gap    = unit(4, "mm"), column_gap = unit(2.5, "mm"),
      rect_gp = gpar(col = "white", lwd = 0.6), border = FALSE,
      row_names_gp    = gpar(fontsize = 12),
      column_names_gp = gpar(fontsize = 11),
      column_names_rot = 60, column_names_max_height = unit(8, "cm"),
      show_heatmap_legend = TRUE,
      heatmap_legend_param = list(
        title      = leg_title,
        title_gp   = gpar(fontsize = 12, fontface = "bold"),
        labels_gp  = gpar(fontsize = 10),
        at         = leg_at, labels = leg_labels,
        legend_height = unit(4.5, "cm"), grid_width = unit(5, "mm")
      ),
      cell_fun = function(j, i, x, y, width, height, fill) {
        v <- value_mat[i, j]; s <- sig_mat[i, j]
        txt <- star_fn(s)
        if (!nzchar(txt)) return()
        grid.text(txt, x, y, gp = gpar(fontsize = 12, fontface = "bold",
          col = if (!is.na(v) && abs(v) >= white_thr) "white" else "black"))
      }
    )
  }

  draw_and_save <- function(ht, fp) {
    n_t <- ncol(ht@matrix); n_s <- nrow(ht@matrix)
    pw_x <- max(20, n_t * 0.30 + 10); ph_x <- max(14, n_s * 0.25 + 8)
    save_heatmap_pdf(ht, fp, width = pw_x, height = ph_x,
                     extra_legend = list(cat_legend_x))
  }

  m_enr_log_x <- signed_log2(m_enr_x)
  ht_enr <- make_ht(
    value_mat = m_enr_x, sig_mat = m_p_x, display_mat = m_enr_log_x,
    col_fun_use = col_fun_opt1, leg_title = "Enrichment",
    leg_at     = signed_log2(c(-1000,-300,-100,0,100,300,1000)),
    leg_labels = c("-1000","-300","-100","0","100","300","1000"),
    white_thr  = 300, star_fn = star_from_p
  )
  draw_and_save(ht_enr,
    file.path(figure_out_dir,
              paste0("Supplementary_SLDSC_FM_all_methods_enr_", suffix, ".pdf")))

  ht_tau <- make_ht(
    value_mat = m_tau_x, sig_mat = m_z_x, display_mat = m_tau_x,
    col_fun_use = tau_col_fun_fm, leg_title = "tau",
    leg_at     = c(-2.0,-1.0,0,0.5,1.0,2.0),
    leg_labels = c("-2.0","-1.0","0","0.5","1.0","2.0"),
    white_thr  = 1.0, star_fn = star_from_z
  )
  draw_and_save(ht_tau,
    file.path(figure_out_dir,
              paste0("Supplementary_SLDSC_FM_all_methods_tau_", suffix, ".pdf")))
}

render_g1(drop_cats = character(),                                  suffix = "all")
render_g1(drop_cats = c("Behavioral","Neuroimaging","Psychiatric"), suffix = "main")


[g1:all] 64 studies x 96 traits



rs_x
ColocBoost  Multi-ctx univariate 
         4          2         58 


Saved: /mnt/lustre/lab/gwang/users/rf2872/Work/codes/xqtl-paper/main_text/2_single_context_cis/figure_data/Supplementary_SLDSC_FM_all_methods_enr_all.pdf



Saved: /mnt/lustre/lab/gwang/users/rf2872/Work/codes/xqtl-paper/main_text/2_single_context_cis/figure_data/Supplementary_SLDSC_FM_all_methods_tau_all.pdf



[g1:main] 64 studies x 49 traits



rs_x
ColocBoost  Multi-ctx univariate 
         4          2         58 


Saved: /mnt/lustre/lab/gwang/users/rf2872/Work/codes/xqtl-paper/main_text/2_single_context_cis/figure_data/Supplementary_SLDSC_FM_all_methods_enr_main.pdf



Saved: /mnt/lustre/lab/gwang/users/rf2872/Work/codes/xqtl-paper/main_text/2_single_context_cis/figure_data/Supplementary_SLDSC_FM_all_methods_tau_main.pdf



## Landmark traits — univariate (main tau*, supplementary enrichment)

Curated literature-driven trait whitelist (AD/brain-focused). Renders to
`figure_data/`:
- `main_SLDSC_FM_landmark_univariate_tau.pdf`        (main figure: tau*)
- `Supplementary_SLDSC_FM_landmark_univariate_enr.pdf` (supplementary: enrichment)


In [6]:
landmark_traits <- tibble::tribble(
  ~trait,                                                ~category,    ~citation,
  "AD_Bellenguez_EADB_hg38_sorted",                      "AD",         "Bellenguez 2022 Nat Genet (EADB)",
  "AD_Bellenguez_buildGRCh38",                           "AD",         "Bellenguez 2022 Nat Genet (GRCh38)",
  "AD_Kunkle_etal_Stage1_results_hg38.sorted.munged",    "AD",         "Kunkle 2019 Nat Genet (IGAP Stage 1)",
  "AD_Wightman_Excluding23andMe",                        "AD",         "Wightman 2021 Nat Genet (excl. 23andMe)",
  "AD_Wightman_ExcludingUKB23andMe",                     "AD",         "Wightman 2021 Nat Genet (excl. UKB+23andMe)",
  "AD_Wightman_full_2021",                               "AD",         "Wightman 2021 Nat Genet (full)",
  "PASS_Alzheimers_Jansen2019",                          "AD",         "Jansen 2019 Nat Genet (UKB proxy-AD)",
  "UKB.Mono.BOLT",                                       "Blood",      "TREM2/CD33/MS4A/PLCG2/ABI3 expressed in monocytes/microglia",
  "UKB.Lym.BOLT",                                        "Blood",      "Gate 2020 Nature: T-cell infiltration in AD brain",
  "UKB.Neutro.BOLT",                                     "Blood",      "Zenaro 2015 Nat Med: neutrophil extravasation in AD",
  "PASS_Multiple_sclerosis",                             "Autoimmune", "IMSGC; primary CNS autoimmune disease",
  "PASS_IBD_deLange2017",                                "Autoimmune", "deLange 2017; gut-brain axis, IBD-dementia link",
  "PASS_CD_deLange2017",                                 "Autoimmune", "deLange 2017; NOD2 myeloid signal",
  "PASS_Rheumatoid_Arthritis",                           "Autoimmune", "Wallin 2012: RA associated with dementia risk"
)
landmark_traits$trait_norm <- sub("\\.parquet$", "", landmark_traits$trait)
message("[landmark] traits: ", nrow(landmark_traits),
        " (AD=", sum(landmark_traits$category == "AD"),
        ", Blood=", sum(landmark_traits$category == "Blood"),
        ", Autoimmune=", sum(landmark_traits$category == "Autoimmune"), ")")

fwrite(landmark_traits,
       file.path(figure_data_dir, "SLDSC_FM_landmark_trait_selection.tsv"),
       sep = "\t")

build_landmark_heatmap <- function(value_tbl, loci_tbl, landmark_tbl,
                                    method_levels,
                                    col_fun_use, leg_title, leg_at, leg_labels,
                                    white_thr, star_fn, log_transform = FALSE) {
  studies_use <- unique(loci_tbl$study)
  traits_keep <- landmark_tbl$trait_norm

  plot_tbl <- value_tbl %>%
    filter(study %in% studies_use, trait %in% traits_keep) %>%
    distinct(study, trait, .keep_all = TRUE)
  if (nrow(plot_tbl) == 0L) stop("No rows match the landmark traits.")

  m_val <- plot_tbl %>% select(study, trait, value) %>%
    pivot_wider(names_from = trait, values_from = value) %>%
    column_to_rownames("study") %>% as.matrix()
  m_sig <- plot_tbl %>% select(study, trait, sig) %>%
    pivot_wider(names_from = trait, values_from = sig) %>%
    column_to_rownames("study") %>% as.matrix()

  miss_cols <- setdiff(traits_keep, colnames(m_val))
  if (length(miss_cols)) {
    pad <- matrix(NA_real_, nrow = nrow(m_val), ncol = length(miss_cols),
                  dimnames = list(rownames(m_val), miss_cols))
    m_val <- cbind(m_val, pad); m_sig <- cbind(m_sig, pad)
  }
  col_ord <- landmark_tbl$trait_norm
  m_val <- m_val[, col_ord, drop = FALSE]
  m_sig <- m_sig[, col_ord, drop = FALSE]

  row_meth <- loci_tbl %>% distinct(study, method)
  rs <- row_meth$method[match(rownames(m_val), row_meth$study)]
  rs_fac <- factor(rs, levels = method_levels)
  ord_idx <- order(as.integer(rs_fac), rownames(m_val))
  m_val <- m_val[ord_idx, , drop = FALSE]
  m_sig <- m_sig[ord_idx, , drop = FALSE]
  rs_fac <- rs_fac[ord_idx]

  col_anno <- tibble(trait = colnames(m_val)) %>%
    left_join(landmark_tbl %>% select(trait_norm, category),
              by = c("trait" = "trait_norm"))
  col_anno$category <- factor(col_anno$category,
                              levels = c("AD","Blood","Autoimmune"))
  cat_col <- make_category_palette(levels(col_anno$category))
  ha_top  <- HeatmapAnnotation(
    Bar = anno_block(gp = gpar(fill = cat_col, col = NA),
                     height = unit(3, "mm")),
    show_annotation_name = FALSE
  )

  display_mat <- if (log_transform) signed_log2(m_val) else m_val
  n_t <- ncol(display_mat); n_s <- nrow(display_mat)

  ht <- Heatmap(
    display_mat, name = leg_title, col = col_fun_use, na_col = "grey90",
    width  = unit(n_t * 3.8, "mm"),
    height = unit(n_s * 5.4, "mm"),
    cluster_rows = FALSE, cluster_columns = FALSE,
    show_row_dend = FALSE, show_column_dend = FALSE,
    row_split = rs_fac, cluster_row_slices = FALSE,
    column_split = col_anno$category, cluster_column_slices = FALSE,
    top_annotation = ha_top,
    row_labels     = clean_qtl_row_labels(rownames(display_mat)),
    column_labels  = prettify_labels(colnames(display_mat)),
    show_row_names = TRUE,
    row_names_max_width = max_text_width(clean_qtl_row_labels(rownames(display_mat)),
                                          gp = gpar(fontsize = 8.5)) +
                          unit(6, "mm"),
    row_title_gp   = gpar(fontsize = 9, fontface = "bold"),
    row_title_rot  = 90,
    column_title   = NULL,
    row_gap    = unit(4, "mm"), column_gap = unit(2.5, "mm"),
    rect_gp = gpar(col = "white", lwd = 0.6), border = FALSE,
    row_names_gp    = gpar(fontsize = 8.5),
    column_names_gp = gpar(fontsize = 8.5),
    column_names_rot = 60, column_names_max_height = unit(8, "cm"),
    show_heatmap_legend = TRUE,
    heatmap_legend_param = list(
      title      = leg_title,
      title_gp   = gpar(fontsize = 9, fontface = "bold"),
      labels_gp  = gpar(fontsize = 8),
      at         = leg_at, labels = leg_labels,
      legend_height = unit(3.2, "cm"), grid_width = unit(3.5, "mm")
    ),
    cell_fun = function(j, i, x, y, width, height, fill) {
      v <- m_val[i, j]; s <- m_sig[i, j]
      txt <- star_fn(s)
      if (!nzchar(txt)) return()
      grid.text(txt, x, y, gp = gpar(fontsize = 8.5, fontface = "bold",
        col = if (!is.na(v) && abs(v) >= white_thr) "white" else "black"))
    }
  )
  list(ht = ht, n_t = n_t, n_s = n_s)
}

cat_legend_lm <- Legend(
  labels = c("AD","Blood","Autoimmune"),
  legend_gp = gpar(fill = make_category_palette(c("AD","Blood","Autoimmune"))),
  title = "Category", title_gp = gpar(fontsize = 11, fontface = "bold"),
  labels_gp = gpar(fontsize = 10),
  grid_height = unit(4, "mm"), grid_width = unit(4, "mm")
)

draw_save_landmark <- function(res, fp) {
  pw_l <- max(5.8, res$n_t * 0.12 + 4.1)
  ph_l <- max(8.0, res$n_s * 0.20 + 4.0)
  save_heatmap_pdf(res$ht, fp, width = pw_l, height = ph_l,
                   extra_legend = list(cat_legend_lm))
}

# ── Univariate filter (ROSMAP cohort only; mega for single-cell; drop trans
#    and DeJager/Kellis PI variants) ─────────────────────────────────────────
loci_univ <- loci_fm_all %>%
  filter(method == "univariate",
         grepl("^ROSMAP_",   study),
         !grepl("_adjusted", study),
         !grepl("_trans_",   study),
         !grepl("_(DeJager|Kellis)$", study))
message("[landmark] univariate studies after filter: ", nrow(loci_univ))
print(loci_univ %>% arrange(study) %>% select(study, loci_type, loci_group))

enr_long <- enr_fm_all %>% transmute(study, trait,
                                     value = Enrichment, sig = Enrichment_p)
tau_long <- tau_fm_all %>% transmute(study, trait,
                                     value = tau_star,   sig = tau_z)

res_univ_enr <- build_landmark_heatmap(
  enr_long, loci_univ, landmark_traits,
  method_levels = "univariate",
  col_fun_use = col_fun_opt1, leg_title = "Enrichment",
  leg_at     = signed_log2(c(-1000,-300,-100,0,100,300,1000)),
  leg_labels = c("-1000","-300","-100","0","100","300","1000"),
  white_thr = 300, star_fn = star_from_p, log_transform = TRUE
)
draw_save_landmark(res_univ_enr,
  file.path(figure_out_dir, "Supplementary_SLDSC_FM_landmark_univariate_enr.pdf"))

res_univ_tau <- build_landmark_heatmap(
  tau_long, loci_univ, landmark_traits,
  method_levels = "univariate",
  col_fun_use = tau_col_fun_fm, leg_title = "tau",
  leg_at     = c(-2.0,-1.0,0,0.5,1.0,2.0),
  leg_labels = c("-2.0","-1.0","0","0.5","1.0","2.0"),
  white_thr = 1.0, star_fn = star_from_z, log_transform = FALSE
)
draw_save_landmark(res_univ_tau,
  file.path(figure_out_dir, "main_SLDSC_FM_landmark_univariate_tau.pdf"))


[landmark] traits: 14 (AD=7, Blood=3, Autoimmune=4)



[landmark] univariate studies after filter: 17



# A tibble: 17 × 3
   study                               loci_type loci_group      
   <chr>                               <chr>     <chr>           
 1 ROSMAP_eQTL_AC                      eQTL      univariate_eQTL 
 2 ROSMAP_eQTL_Ast_mega                eQTL      univariate_eQTL 
 3 ROSMAP_eQTL_DLPFC                   eQTL      univariate_eQTL 
 4 ROSMAP_eQTL_Exc_mega                eQTL      univariate_eQTL 
 5 ROSMAP_eQTL_Inh_mega                eQTL      univariate_eQTL 
 6 ROSMAP_eQTL_Mic_mega                eQTL      univariate_eQTL 
 7 ROSMAP_eQTL_OPC_mega                eQTL      univariate_eQTL 
 8 ROSMAP_eQTL_Oli_mega                eQTL      univariate_eQTL 
 9 ROSMAP_eQTL_PCC                     eQTL      univariate_eQTL 
10 ROSMAP_eQTL_monocyte                eQTL      univariate_eQTL 
11 ROSMAP_gpQTL_DLPFC_Klein_unadjusted gpQTL     univariate_gpQTL
12 ROSMAP_haQTL_DLPFC                  haQTL     univariate_haQTL
13 ROSMAP_mQTL_DLPFC                   mQTL      univaria

Saved: /mnt/lustre/lab/gwang/users/rf2872/Work/codes/xqtl-paper/main_text/2_single_context_cis/figure_data/Supplementary_SLDSC_FM_landmark_univariate_enr.pdf



Saved: /mnt/lustre/lab/gwang/users/rf2872/Work/codes/xqtl-paper/main_text/2_single_context_cis/figure_data/main_SLDSC_FM_landmark_univariate_tau.pdf



## Landmark traits — ColocBoost + Multi-context

Outputs go to `3_multi_context/figure_data/`:
- `main_SLDSC_FM_landmark_coloc_multictx_tau.pdf`        (main figure: tau*)
- `Supplementary_SLDSC_FM_landmark_coloc_multictx_enr.pdf` (supplementary: enrichment)


In [7]:
loci_cm <- loci_fm_all %>%
  filter(method %in% c("colocboost", "multicontext")) %>%
  mutate(method = recode_method(method))
message("[landmark] ColocBoost + Multi-ctx studies: ", nrow(loci_cm))
print(loci_cm %>% count(method))

res_cm_enr <- build_landmark_heatmap(
  enr_long, loci_cm, landmark_traits,
  method_levels = c("ColocBoost", "Multi-ctx"),
  col_fun_use = col_fun_opt1, leg_title = "Enrichment",
  leg_at     = signed_log2(c(-1000,-300,-100,0,100,300,1000)),
  leg_labels = c("-1000","-300","-100","0","100","300","1000"),
  white_thr = 300, star_fn = star_from_p, log_transform = TRUE
)
draw_save_landmark(res_cm_enr,
  file.path(figure_out_dir, "Supplementary_SLDSC_FM_landmark_coloc_multictx_enr.pdf"))

res_cm_tau <- build_landmark_heatmap(
  tau_long, loci_cm, landmark_traits,
  method_levels = c("ColocBoost", "Multi-ctx"),
  col_fun_use = tau_col_fun_fm, leg_title = "tau",
  leg_at     = c(-2.0,-1.0,0,0.5,1.0,2.0),
  leg_labels = c("-2.0","-1.0","0","0.5","1.0","2.0"),
  white_thr = 1.0, star_fn = star_from_z, log_transform = FALSE
)
draw_save_landmark(res_cm_tau,
  file.path(figure_out_dir, "main_SLDSC_FM_landmark_coloc_multictx_tau.pdf"))


[landmark] ColocBoost + Multi-ctx studies: 6



# A tibble: 2 × 2
  method         n
  <chr>      <int>
1 ColocBoost     4
2 Multi-ctx      2


Saved: /mnt/lustre/lab/gwang/users/rf2872/Work/codes/xqtl-paper/main_text/3_multi_context/figure_data/Supplementary_SLDSC_FM_landmark_coloc_multictx_enr.pdf



Saved: /mnt/lustre/lab/gwang/users/rf2872/Work/codes/xqtl-paper/main_text/3_multi_context/figure_data/main_SLDSC_FM_landmark_coloc_multictx_tau.pdf

